# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, leveraging its Croissant JSON-LD schema and referencing all dataset entities via their `@id` identifiers.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

**Note:** All references to record sets, fields, and columns use their unique `@id` values for clarity and reproducibility.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
ds = mlc.Dataset(croissant_url)

# Access the dataset metadata as a native object (avoid dict subscripting)
metadata = ds.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")


## 2. Data Overview
Review available record sets, their fields, and refer to all by their `@id` identifiers.

In [ ]:
# List all record sets and their fields by `@id`
record_sets = ds.record_sets
if not record_sets:
    print("No record sets found in the dataset schema.")
else:
    for rs in record_sets:
        print(f"Record Set Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
        print()

## 3. Data Extraction
Load data from each record set into a DataFrame.

> **Note:** All record set and field references below use the `@id` property for reproducibility. Set the variable `record_set_ids` to the record set `@id`s from the overview.

In [ ]:
# Gather record set @id values
record_set_ids = [rs.id for rs in ds.record_sets]

# Create a DataFrame for each record set
dfs = {}
for rs_id in record_set_ids:
    recs = list(ds.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(recs)

# Print sample columns and preview for the first available record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in record set {first_rs_id}:")
    print(dfs[first_rs_id].columns.tolist())
    display(dfs[first_rs_id].head(5))
else:
    print("No record sets to extract data from.")

## 4. Exploratory Data Analysis (EDA)
We'll process and analyze the main record set. Operations include filtering, normalizing a numeric field, and grouping by a categorical field (referenced by their `@id`).

In [ ]:
# Identify a numeric and a categorical field by `@id`
record_set_id = first_rs_id  # Use the first record set as example

# Find numeric fields (fields where data_type is Float or Integer)
rs_obj = [rs for rs in ds.record_sets if rs.id == record_set_id][0] if ds.record_sets else None
if rs_obj:
    numeric_fields = [field for field in rs_obj.fields if field.data_type in ("schema:Float", "schema:Integer", "Integer", "Float", "Number")]
    categorical_fields = [field for field in rs_obj.fields if field.data_type in ("schema:Text", "Text", "String")]
else:
    numeric_fields = []
    categorical_fields = []

if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    print("No numeric fields found.")

if categorical_fields:
    group_field_id = categorical_fields[0].id
    print(f"Grouping by field @id: {group_field_id}")
else:
    group_field_id = None

df = dfs[record_set_id]

# EDA: Filter, normalize, and group (handle non-numeric gracefully)
if numeric_fields and numeric_field_id in df.columns:
    # Try to convert column to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered.head(5))

    normalized_col = f"{numeric_field_id}_normalized"
    filtered[normalized_col] = (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered[[numeric_field_id, normalized_col]].head(5))

    if group_field_id and group_field_id in filtered.columns:
        grouped = filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index(name="mean_value")
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped.head())
else:
    print("No suitable numeric field for EDA in the current record set.")

## 5. Visualization
Visualize the distribution of a numeric field, and its mean per group (if groupable).

> All axes and legends reference the column `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    _ = sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.xlabel(f"{numeric_field_id}")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        _ = sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xlabel(f"{group_field_id}")
        plt.ylabel(f"{numeric_field_id}")
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion

- We successfully loaded the dataset and its metadata using the `mlcroissant` library and the Croissant JSON-LD schema.
- All exploration referenced record sets and fields by their unique `@id`.
- We previewed data, performed sample EDA (filtering, normalization, grouping), and visualized distributions.

Refer to the dataset schema for authoritative definitions and always use `@id` for referencing entities when processing FAIR² datasets.